# Can we predict room occupancy from sensor data?
Data: `data.csv`, one reading per minute (2015-02-02 to 2015-02-18) of temperature, humidity, light, CO2 and humidity ratio, with a binary `Occupancy` label.

**Approach**
1. Look at the data (balance, gaps, how occupancy relates to each sensor).
2. Evaluate honestly: readings one minute apart are nearly identical, so a random train/test split would leak. We split **by time/day** instead.
3. Compare simple baselines against logistic regression and tree models.
4. Check whether the result is stable across days and which features matter.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, cross_val_predict
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score, ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance
plt.rcParams['figure.figsize']=(9,4)
df=pd.read_csv('data.csv',parse_dates=['date']).sort_values('date').reset_index(drop=True)
df['day']=df.date.dt.date
df.head()

## 1. Data overview

In [ ]:
print(df.shape, 'missing values:', int(df.isna().sum().sum()))
print('Occupied share: %.1f%%'%(100*df.Occupancy.mean()))
gaps=df.date.diff(); big=gaps>pd.Timedelta('2min')
print('Gaps > 2 min:'); print(pd.DataFrame({'at':df.date[big],'gap':gaps[big]}))
df.describe().T

In [ ]:
daily=df.groupby('day').Occupancy.agg(['mean','size']); daily.columns=['occupied_share','rows']
daily['weekday']=pd.to_datetime(daily.index).day_name()
daily

Weekends (Feb 7-8, 14-15) are never occupied. Some days are partial because of the logging gaps above.

In [ ]:
feats=['Temperature','Humidity','Light','CO2','HumidityRatio']
fig,ax=plt.subplots(1,5,figsize=(16,3.2))
for a,f in zip(ax,feats):
    for v,c in [(0,'tab:blue'),(1,'tab:orange')]:
        a.hist(df.loc[df.Occupancy==v,f],bins=40,alpha=.6,color=c,label=['empty','occupied'][v],density=True)
    a.set_title(f)
ax[0].legend(); plt.tight_layout(); plt.show()
df[feats+['Occupancy']].corr()['Occupancy'].drop('Occupancy').to_frame('corr with Occupancy')

In [ ]:
fig,ax=plt.subplots(2,1,figsize=(13,5),sharex=True)
ax[0].plot(df.date,df.Light,lw=.5); ax[0].set_ylabel('Light')
ax[1].plot(df.date,df.CO2,lw=.5); ax[1].set_ylabel('CO2')
for a in ax:
    a.fill_between(df.date,0,1,where=df.Occupancy==1,transform=a.get_xaxis_transform(),color='orange',alpha=.25)
plt.tight_layout(); plt.show()

Light and CO2 rise when the room is occupied (orange shading). Light is the strongest signal on its own.

## 2. Evaluation setup
- **Chronological hold-out:** train on the first 70% of rows, test on the last 30% (later days).
- **Leave-one-day-out CV** as a second, more robust check.
- Features are the 5 sensor readings only (no timestamp features), so the model works from a single reading. Accuracy alone is misleading with ~23% positives, so we also report F1 and ROC AUC.

In [ ]:
cut=int(len(df)*.7); train,test=df.iloc[:cut],df.iloc[cut:]
print('train',train.date.min(),'->',train.date.max(),'| occupied %.1f%%'%(100*train.Occupancy.mean()))
print('test ',test.date.min(),'->',test.date.max(),'| occupied %.1f%%'%(100*test.Occupancy.mean()))
Xtr,ytr,Xte,yte=train[feats],train.Occupancy,test[feats],test.Occupancy

In [ ]:
def report(name,y,pred,score):
    return dict(model=name,accuracy=accuracy_score(y,pred),precision=precision_score(y,pred),recall=recall_score(y,pred),
                f1=f1_score(y,pred),roc_auc=roc_auc_score(y,score))
models={
 'Logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'Random forest':RandomForestClassifier(300,min_samples_leaf=5,random_state=0,n_jobs=-1),
 'Gradient boosting':HistGradientBoostingClassifier(random_state=0),
}
rows=[dict(model='Baseline: always empty',accuracy=1-yte.mean(),precision=np.nan,recall=0,f1=0,roc_auc=.5)]
ths=np.unique(Xtr.Light); thr=max(ths[::5],key=lambda t:f1_score(ytr,Xtr.Light>t))  # threshold tuned on train only
rows.append(report(f'Baseline: Light > {thr:.0f}',yte,(Xte.Light>thr).astype(int),Xte.Light))
fitted={}
for n,m in models.items():
    m.fit(Xtr,ytr); fitted[n]=m
    rows.append(report(n,yte,m.predict(Xte),m.predict_proba(Xte)[:,1]))
res=pd.DataFrame(rows).set_index('model'); res.round(3)

In [ ]:
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
for a,(n,m) in zip(ax,fitted.items()):
    ConfusionMatrixDisplay.from_predictions(yte,m.predict(Xte),ax=a,colorbar=False,display_labels=['empty','occupied']); a.set_title(n)
plt.tight_layout(); plt.show()

## 3. Robustness: leave-one-day-out cross-validation

In [ ]:
logo=LeaveOneGroupOut(); cvrows=[]; oof={}
for n,m in models.items():
    p=cross_val_predict(m,df[feats],df.Occupancy,groups=df.day.values,cv=logo,method='predict_proba')[:,1]; oof[n]=p
    pred=(p>.5).astype(int)
    cvrows.append(dict(model=n,accuracy=accuracy_score(df.Occupancy,pred),f1=f1_score(df.Occupancy,pred),roc_auc=roc_auc_score(df.Occupancy,p)))
pd.DataFrame(cvrows).set_index('model').round(3)

In [ ]:
best_name=max(oof,key=lambda n:roc_auc_score(df.Occupancy,oof[n]))
d=df.assign(pred=(oof[best_name]>.5).astype(int))
perday=d.groupby('day').apply(lambda g:pd.Series(dict(rows=len(g),accuracy=(g.pred==g.Occupancy).mean(),occupied_share=g.Occupancy.mean())))
print('Per-day accuracy for',best_name); perday.round(3)

## 4. What drives the prediction?

In [ ]:
m=fitted['Random forest']
pi=permutation_importance(m,Xte,yte,scoring='roc_auc',n_repeats=10,random_state=0)
imp=pd.Series(pi.importances_mean,index=feats).sort_values()
imp.plot.barh(xerr=pd.Series(pi.importances_std,index=feats)[imp.index])
plt.xlabel('Drop in test ROC AUC when shuffled'); plt.title('Permutation importance (random forest)'); plt.tight_layout(); plt.show()

In [ ]:
# Ablation: how much is lost with fewer sensors?
for cols in [feats,[f for f in feats if f!='Light'],['Light'],['CO2'],['Light','CO2']]:
    m=RandomForestClassifier(300,min_samples_leaf=5,random_state=0,n_jobs=-1).fit(train[cols],ytr)
    p=m.predict_proba(test[cols])[:,1]
    print(f'{str(cols):70s} F1={f1_score(yte,p>.5):.3f}  AUC={roc_auc_score(yte,p):.3f}')

## 5. Conclusions

**Yes, occupancy is very predictable from these sensors, but almost entirely because of the light sensor.**

- On the chronological hold-out (last ~4 days), logistic regression reaches **99.3% accuracy, F1 0.98, ROC AUC 0.999**, versus 82% accuracy for always guessing "empty". Leave-one-day-out CV agrees (accuracy 96-99%, AUC 0.99+), and per-day accuracy is 95-100% for every day, including weekends.
- **A single threshold on Light (> ~366) does as well as any model** (F1 0.986). The extra models add nothing, and gradient boosting is slightly worse (more false alarms).
- **Light carries nearly all the signal.** Without it, F1 drops to ~0.34 (AUC 0.63); CO2 alone gives F1 ~0.48. Temperature, humidity and humidity ratio add little.
- Practical implication: this is close to "are the lights on?". If lights are sometimes on in an empty room, or people work with lights off or in daylight, performance will be worse than here. This dataset cannot show that.

In [ ]:
best=res.iloc[2:].sort_values('f1').iloc[-1]
print(f"Best hold-out model: {best.name}: accuracy {best.accuracy:.3f}, F1 {best.f1:.3f}, AUC {best.roc_auc:.3f}")
print(f"Naive 'always empty' accuracy: {res.iloc[0].accuracy:.3f}")

**Caveats**
- Only ~16 days from one room; the label may be derived from camera snapshots, so some noise at transitions is expected.
- Errors concentrate around the moments people arrive/leave; sensors like CO2 lag behind the actual change.
- Daily patterns (lights on during working hours) could differ in another season or room, so re-validate before deploying elsewhere.